In [2]:
import pandas as pd

In [147]:
import numpy as np
import json

_k = 3
_ret = 'e5'
_dataset = 'nq_test'

experiment_name = f'{_dataset}_{_ret}'

# f = open(f'./log_prob_temp_res/{experiment_name}_12.json') # for individual perp w/o query as preamble
f = open(f'./log_prob_temp_res/individual_with_query/{experiment_name}_20.json') # for individual perp w/ query as preamble
# f = open(f'./log_prob_temp_res/individual/{experiment_name}_20.json') # for individual perp w/ query as preamble
perp_json = json.load(f)
f.close()

retr_res = pd.read_csv(f'../../rag_utility/res/{_ret}_{_dataset}.csv')

max_score_dict = {}
avg_score_dict = {}

for _qid, _record in perp_json.items():
    _value_list = []
    for _rank, _value in _record.items():
        if(int(_rank) >= _k):
            break
        _value_list.append(_value)
        max_score_dict.update({str(_qid): np.exp(_value_list).max()})
        avg_score_dict.update({str(_qid): np.exp(_value_list).mean()})

In [148]:
import json
import numpy as np

def tool_for_aggregating_dl_performance(x):
    # the same as performLoader
    scores = []
    for answer_eval in x[1]['0'].values():
        scores.append(max(answer_eval['qrel_2']['f1']['max'], answer_eval['qrel_3']['f1']['max']))
    return np.mean(scores)

if('nq' in _dataset):
    _dataset_test, _prefix, _suffix, call_num = [_dataset], 'short', 'concise', 1
else:
    _dataset_test, _prefix, _suffix, call_num = ['19', '20'], 'random', 'prompt1', 5

zero_evals, k_evals, k_gens = {}, {}, {}

for _d in _dataset_test:
    f = open(f'../../rag_utility/eval_results/{_prefix}_answers_0shot_{call_num}calls_0_0_bm25_dl_{_d}_{_suffix}_eval.json')
    if(zero_evals=={}):
        zero_evals = json.load(f)
    else:
        zero_evals.update(json.load(f))
    f.close()
    
    f = open(f'../../rag_utility/eval_results/{_prefix}_answers_{_k}shot_{call_num}calls_1_0_{_ret}_dl_{_d}_{_suffix}_eval.json')
    if(k_evals=={}):
        k_evals = json.load(f)
    else:
        k_evals.update(json.load(f))
    f.close()
    
    f = open(f'../../rag_utility/gen_results/{_prefix}_answers_{_k}shot_{call_num}calls_1_0_{_ret}_dl_{_d}_{_suffix}.json')
    if(k_gens=={}):
        k_gens = json.load(f)
    else:
        k_gens.update(json.load(f))
    f.close()

test_res = retr_res[['qid', 'query']].drop_duplicates().copy()
test_res.qid = test_res.qid.astype('str')
test_res = test_res[test_res.qid.isin(max_score_dict.keys())]
test_res['avg_quality'] = test_res.qid.apply(lambda x: avg_score_dict[x])
test_res['max_quality'] = test_res.qid.apply(lambda x: max_score_dict[x])

if('nq' in _dataset):
    base_f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in zero_evals.items() if ('0' in item[1].keys())}
    f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in k_evals.items() if ('0' in item[1].keys())}
else:
    base_f1_dict = {item[0]: tool_for_aggregating_dl_performance(item) for item in zero_evals.items() if ('0' in item[1].keys())}
    f1_dict = {item[0]: tool_for_aggregating_dl_performance(item) for item in k_evals.items() if ('0' in item[1].keys())}

kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items() if ('0' in item[1].keys())}

test_res = test_res[test_res.qid.astype('str').isin(f1_dict.keys())]
test_res['f1'] = test_res.qid.apply(lambda x: f1_dict[str(x)])
test_res['utility'] = test_res.qid.apply(lambda x: f1_dict[str(x)]-base_f1_dict[str(x)])

test_res = test_res.dropna(axis='index')
print(test_res.shape)
test_res.head(3)

(3610, 6)


,qid,query,avg_quality,max_quality,f1,utility
0,test_0,who got the first nobel prize in physics,0.303101,0.351376,0.8,-0.2
120,test_1,when is the next deadpool movie being released,0.093941,0.155163,0.0,0.0
240,test_2,which mode is used for short wave broadcast se...,0.055426,0.060172,0.0,0.0


In [149]:
from scipy import stats

print(stats.spearmanr(test_res.avg_quality, test_res.f1)[0], stats.kendalltau(test_res.avg_quality, test_res.f1)[0])
print(stats.spearmanr(test_res.avg_quality, test_res.utility)[0], stats.kendalltau(test_res.avg_quality, test_res.utility)[0])

0.12837113565818076 0.09704254753059995
0.03835629454005153 0.027892286982655005


In [150]:
print(stats.spearmanr(test_res.max_quality, test_res.f1)[0], stats.kendalltau(test_res.max_quality, test_res.f1)[0])
print(stats.spearmanr(test_res.max_quality, test_res.utility)[0], stats.kendalltau(test_res.max_quality, test_res.utility)[0])

0.11507776215378136 0.08687817992341759
0.03379663522436141 0.02450857862076839
